# Real-Time Object Detection with MediaPipe and OpenCV

Este notebook implementa a detecção de objetos em tempo real usando a webcam, utilizando a biblioteca **Google MediaPipe** e **OpenCV**.

### Pré-requisitos
Certifique-se de ter as bibliotecas instaladas:
```bash
pip install mediapipe opencv-python numpy
```

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time

## Configuração do Detector de Objetos

Vamos usar o modelo `efficientdet_lite0.tflite` que já está no diretório.

In [3]:
model_path = 'models/efficientdet_lite0.tflite'

BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    score_threshold=0.5,
    running_mode=VisionRunningMode.VIDEO
)

detector = ObjectDetector.create_from_options(options)

## Loop de Captura da Webcam

Este bloco abre a webcam, processa cada quadro com o MediaPipe e desenha as caixas delimitadoras nos objetos detectados.

In [4]:
def draw_landmarks(image, detection_result):
    """Desenha as caixas e labels na imagem."""
    for detection in detection_result.detections:
        # Coordenadas da caixa
        bbox = detection.bounding_box
        start_point = bbox.origin_x, bbox.origin_y
        end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
        
        # Desenha a caixa
        cv2.rectangle(image, start_point, end_point, (0, 255, 0), 2)

        # Nome da categoria e score
        category = detection.categories[0]
        category_name = category.category_name
        probability = round(category.score, 2)
        result_text = f"{category_name} ({probability})"
        text_location = (bbox.origin_x, bbox.origin_y - 10)
        
        cv2.putText(image, result_text, text_location, cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2)

    return image

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro: Não foi possível abrir a webcam.")
else:
    print("Webcam aberta com sucesso! Pressione 'q' para sair.")
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Falha ao capturar imagem.")
            break

        # Converte para RGB (exigido pelo MediaPipe)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Timestamp em milissegundos
        timestamp_ms = int(time.time() * 1000)

        # Realiza a detecção
        detection_result = detector.detect_for_video(mp_image, timestamp_ms)

        # Desenha resultados
        annotated_frame = draw_landmarks(frame.copy(), detection_result)

        # Mostra o FPS (opcional)
        cv2.imshow('MediaPipe Object Detection', annotated_frame)

        # Encerra o loop se 'q' for pressionado
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam encerrada.")

Webcam aberta com sucesso! Pressione 'q' para sair.
Webcam encerrada.
